# GeoPandas と folium で学ぶ地理空間データ分析 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
地理空間データ（地図データ）を扱うライブラリ **GeoPandas** と、対話的な地図を描く **folium** の基礎を学ぶためのチュートリアルです。
題材として、同じフォルダにある東海 4 県（岐阜・静岡・愛知・三重）の境界データ `tokai4_prefs.geojson` を使います。

## 対象者
- pandas の DataFrame の基本操作（列の選択、merge、groupby）を理解している方
- 地域経済・都市経済・公共政策など、「場所」に関わるデータを分析したい方
- 地図データ（GeoJSON、座標参照系）を初めて扱う方

## このチュートリアルで学ぶこと
1. 地理空間データとは（点・線・面、GeoJSON、GeoDataFrame）
2. geometry を調べる
3. 属性データの結合
4. 座標参照系（CRS）と面積・人口密度
5. 重心と距離
6. 点データと空間結合（どの県に属するか）
7. バッファと空間演算
8. matplotlib による地図（コロプレス図）
9. folium による対話的な地図
10. GeoJSON の書き出しと読み込み

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 各章の最後に **練習問題** があります。解答欄に自分で書いてから、折りたたみの解答例を開いて確認しましょう。
- このノートブックは `tokai4_prefs.geojson` と同じフォルダ（`python/folium/`）に置いたまま実行してください。

---
## 0. 環境準備（JupyterLite 用）

GeoPandas・Shapely・pyproj は JupyterLite（Pyodide）に同梱されています。folium と日本語フォント
（`japanize-matplotlib-jlite`）は PyPI から取得します。初回は数十秒かかることがあります。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["geopandas", "shapely", "pyproj", "matplotlib", "folium", "japanize-matplotlib-jlite"])
except ImportError:
    pass

In [ ]:
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォント（plt の後に import）
import numpy as np
import pandas as pd
import shapely
import folium

print("GeoPandas:", gpd.__version__)
print("Shapely  :", shapely.__version__)
print("folium   :", folium.__version__)

---
## 1. 地理空間データとは

地図上の「もの」は、次の 3 種類の **ジオメトリ（geometry）** で表されます。

| 種類 | 例 | Shapely の型 |
|---|---|---|
| 点（Point） | 駅、店舗、市役所の位置 | `Point` |
| 線（LineString） | 道路、鉄道、河川 | `LineString` |
| 面（Polygon） | 県境、市区町村、商圏 | `Polygon`, `MultiPolygon` |

これらの図形と属性（名前、人口など）をセットにした形式が **GeoJSON** などのファイルで、
GeoPandas はそれを **GeoDataFrame**（= pandas の DataFrame + `geometry` 列）として読み込みます。
DataFrame でできること（列の選択、条件抽出、merge、groupby）はそのまま使えます。

In [ ]:
# 同じフォルダにある GeoJSON を読み込む
pref = gpd.read_file("tokai4_prefs.geojson")

print(type(pref))
print("行数・列数:", pref.shape)
print("列名:", list(pref.columns))
print("座標参照系:", pref.crs)
pref

In [ ]:
# geometry 以外の列は普通の DataFrame と同じように扱える
print(pref[["nam_ja", "nam", "id"]])
print()
print("県名の一覧:", pref["nam_ja"].tolist())

### 練習問題 1

1. `pref` から `nam_ja` が `"愛知県"` の行だけを取り出して表示してください。
2. `pref` の列 `id` を昇順に並べ替えて、`nam_ja` と `id` の 2 列を表示してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
print(pref[pref["nam_ja"] == "愛知県"])

# 2
print(pref.sort_values("id")[["nam_ja", "id"]])
```

</details>

---
## 2. geometry を調べる

`geometry` 列の各要素は Shapely の図形オブジェクトです。種類（`geom_type`）、範囲（`bounds`）、
妥当性（`is_valid`）などを確認できます。

In [ ]:
print(pref.geometry.geom_type)          # 図形の種類
print()
print("全体の範囲 [minx, miny, maxx, maxy]:", pref.total_bounds.round(3))
print()
print(pref.bounds.round(3))              # 県ごとの範囲

In [ ]:
# 1 つの図形を取り出してみる（愛知県）
aichi = pref.loc[pref["nam_ja"] == "愛知県", "geometry"].iloc[0]
print(type(aichi))
print("図形の種類:", aichi.geom_type)
print("妥当な図形か:", aichi.is_valid)
print("ポリゴンの数:", len(aichi.geoms) if aichi.geom_type == "MultiPolygon" else 1)
aichi   # セルの最後に置くと図形が描画される

In [ ]:
# 図形の中に必ず入る代表点（ラベルの配置などに使う）
rep = pref.representative_point()
for name, p in zip(pref["nam_ja"], rep):
    print(f"{name}: 経度 {p.x:.3f}, 緯度 {p.y:.3f}")

### 練習問題 2

1. 静岡県の geometry を取り出し、`geom_type` と `bounds`（範囲）を表示してください。
2. 4 県それぞれについて、`bounds` の東西方向の幅（`maxx - minx`）を計算して表示してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
shizuoka = pref.loc[pref["nam_ja"] == "静岡県", "geometry"].iloc[0]
print(shizuoka.geom_type)
print(shizuoka.bounds)

# 2
b = pref.bounds
width = (b["maxx"] - b["minx"]).round(3)
print(pd.DataFrame({"県": pref["nam_ja"], "東西の幅（度）": width}))
```

</details>

---
## 3. 属性データの結合

地図データには県名しかありません。分析には人口や経済規模などの **属性データ** を結合（merge）します。
ここでは例示用の概算値（2020 年前後の統計をもとにした丸めた値）を DataFrame として用意します。

In [ ]:
stats = pd.DataFrame({
    "nam_ja": ["岐阜県", "静岡県", "愛知県", "三重県"],
    "population": [1_980_000, 3_630_000, 7_550_000, 1_770_000],   # 人口（人・概算）
    "gdp_trillion": [7.9, 17.2, 40.9, 8.3],                        # 県内総生産（兆円・概算）
    "capital": ["岐阜市", "静岡市", "名古屋市", "津市"],
})
print(stats)

In [ ]:
# 県名をキーに結合する。GeoDataFrame.merge の結果は GeoDataFrame のまま
gdf = pref.merge(stats, on="nam_ja")
print(type(gdf))
print(gdf[["nam_ja", "population", "gdp_trillion", "capital"]])

In [ ]:
# 1 人あたり県内総生産（万円）を計算して列に追加
gdf["gdp_per_capita"] = gdf["gdp_trillion"] * 1e12 / gdf["population"] / 1e4
print(gdf[["nam_ja", "gdp_per_capita"]].round(0))

### 練習問題 3

1. 4 県の人口合計と、各県の人口シェア（%）を計算して表示してください。
2. `gdp_trillion` が 10 兆円を超える県だけを `nam_ja` と `gdp_trillion` で表示してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
total = gdf["population"].sum()
print("合計人口:", total)
print((gdf.set_index("nam_ja")["population"] / total * 100).round(1))

# 2
print(gdf.loc[gdf["gdp_trillion"] > 10, ["nam_ja", "gdp_trillion"]])
```

</details>

---
## 4. 座標参照系（CRS）と面積・人口密度

読み込んだデータの座標は **経度・緯度（EPSG:4326, WGS84）** です。経度・緯度は「度」なので、
そのまま面積や距離を計算すると意味のある値になりません（警告も出ます）。

面積・距離を求めるときは、メートル単位の **投影座標系** に変換します。日本では
**平面直角座標系（JGD2011）** がよく使われ、東海地方は **第 7 系（EPSG:6674）** が適しています。
変換は `to_crs()` で行います。

In [ ]:
gdf_m = gdf.to_crs(epsg=6674)          # メートル単位の座標系へ
print(gdf_m.crs)
print()
print("変換後の範囲（m）:", gdf_m.total_bounds.round(0))

In [ ]:
# 面積（m²）→ km² に換算して列に追加
gdf["area_km2"] = gdf_m.area / 1e6
gdf["density"] = gdf["population"] / gdf["area_km2"]     # 人口密度（人/km²）
print(gdf[["nam_ja", "area_km2", "density"]].round(1).sort_values("density", ascending=False))

面積と同じように、`length` で境界線の長さ（周囲長）も求められます。海岸線が複雑な県ほど、面積のわりに長くなります。

In [ ]:
gdf["perimeter_km"] = gdf_m.length / 1000
gdf["compactness"] = gdf["area_km2"] / gdf["perimeter_km"]      # 面積 ÷ 周囲長（大きいほどまとまった形）
print(gdf[["nam_ja", "perimeter_km", "compactness"]].round(1))

`to_crs()` で変換したのは `gdf_m` だけで、元の `gdf` は経度・緯度のままです。
面積や距離は投影後の GeoDataFrame で計算し、結果の数値だけを元のデータに戻す、という使い分けが基本です。

### 練習問題 4

1. 4 県の面積合計（km²）を表示してください。
2. 面積 1 km² あたりの県内総生産（億円/km²）を計算し、大きい順に表示してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
print(round(gdf["area_km2"].sum(), 1), "km²")

# 2
gdf["gdp_per_km2"] = gdf["gdp_trillion"] * 1e4 / gdf["area_km2"]   # 兆円 → 億円
print(gdf[["nam_ja", "gdp_per_km2"]].round(2).sort_values("gdp_per_km2", ascending=False))
```

</details>

---
## 5. 重心と距離

`centroid` で図形の重心（面積の中心）が得られます。これも投影座標系で計算してから、
必要なら経度・緯度に戻します。距離も同様に投影座標系で `distance()` を使います。

In [ ]:
centroids_m = gdf_m.centroid                       # 投影座標系での重心
centroids = centroids_m.to_crs(epsg=4326)          # 経度・緯度に戻す
for name, c in zip(gdf["nam_ja"], centroids):
    print(f"{name}: 経度 {c.x:.3f}, 緯度 {c.y:.3f}")

In [ ]:
# 県の重心どうしの距離（km）の行列
names = gdf["nam_ja"].tolist()
dist = pd.DataFrame(index=names, columns=names, dtype=float)
for i, a in enumerate(centroids_m):
    for j, b in enumerate(centroids_m):
        dist.iloc[i, j] = a.distance(b) / 1000
print(dist.round(1))

### 練習問題 5

1. 岐阜県の重心から最も遠い県を求めてください（`dist` を使います）。
2. 各県の重心から、4 県の重心の平均位置（`centroids_m` の x, y の平均）までの距離（km）を表示してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
print(dist.loc["岐阜県"].idxmax())

# 2
from shapely.geometry import Point
center = Point(centroids_m.x.mean(), centroids_m.y.mean())
for name, c in zip(names, centroids_m):
    print(f"{name}: {c.distance(center) / 1000:.1f} km")
```

</details>

---
## 6. 点データと空間結合

都市の経度・緯度から点の GeoDataFrame を作り（`points_from_xy`）、**空間結合（`sjoin`）** で
「各都市がどの県のポリゴンの中にあるか」を求めます。通常の `merge` がキーの一致で結合するのに対し、
`sjoin` は **位置関係**（`within`: 中にある、`intersects`: 交わる など）で結合します。

In [ ]:
cities = pd.DataFrame({
    "city": ["名古屋市", "豊田市", "静岡市", "浜松市", "岐阜市", "高山市", "津市", "四日市市"],
    "lon": [136.906, 137.156, 138.383, 137.726, 136.762, 137.252, 136.509, 136.625],
    "lat": [35.181, 35.083, 34.976, 34.711, 35.391, 36.146, 34.730, 34.965],
    "city_pop": [2_330_000, 420_000, 690_000, 790_000, 400_000, 85_000, 275_000, 310_000],
})
pts = gpd.GeoDataFrame(cities, geometry=gpd.points_from_xy(cities["lon"], cities["lat"]), crs="EPSG:4326")
print(pts)

In [ ]:
# 空間結合：各都市がどの県の中にあるか
joined = gpd.sjoin(pts, gdf[["nam_ja", "geometry"]], how="left", predicate="within")
print(joined[["city", "nam_ja"]])

In [ ]:
# 県ごとの都市数と都市人口の合計（普通の groupby）
summary = joined.groupby("nam_ja").agg(cities=("city", "count"), city_pop=("city_pop", "sum"))
print(summary)

### 練習問題 6

1. `pts` に新しい都市「岡崎市」（経度 137.174、緯度 34.954、人口 385000）を追加し、どの県に属するか `sjoin` で求めてください。
2. 都市人口（`city_pop`）が県の人口（`population`）に占める割合を県ごとに計算してください（`summary` と `gdf` を結合します）。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
new = gpd.GeoDataFrame({"city": ["岡崎市"], "lon": [137.174], "lat": [34.954], "city_pop": [385_000]},
                       geometry=gpd.points_from_xy([137.174], [34.954]), crs="EPSG:4326")
pts2 = pd.concat([pts, new], ignore_index=True)
print(gpd.sjoin(pts2, gdf[["nam_ja", "geometry"]], predicate="within")[["city", "nam_ja"]].tail(3))

# 2
share = summary.join(gdf.set_index("nam_ja")["population"])
share["ratio"] = (share["city_pop"] / share["population"] * 100).round(1)
print(share)
```

</details>

---
## 7. バッファと空間演算

`buffer(距離)` は図形の周りに一定距離の領域を作ります（商圏や通勤圏の近似に使えます）。
`intersects`、`intersection`、`union_all` などで図形どうしの関係や重なりを計算できます。
これらもメートル単位で行うため、投影座標系に変換してから使います。

In [ ]:
pts_m = pts.to_crs(epsg=6674)
nagoya = pts_m.loc[pts_m["city"] == "名古屋市", "geometry"].iloc[0]
circle = nagoya.buffer(30_000)          # 名古屋から半径 30 km の円

# 円と交わる県
hit = gdf_m[gdf_m.intersects(circle)]
print("半径 30 km 圏と交わる県:", hit["nam_ja"].tolist())

In [ ]:
# 県ごとに、30 km 圏との重なりの面積（km²）
overlap = gdf_m.geometry.intersection(circle).area / 1e6
print(pd.DataFrame({"県": gdf["nam_ja"], "重なり面積 km²": overlap.round(1)}))

In [ ]:
# 4 県をひとつの図形にまとめる（union）
tokai = gdf_m.union_all()
print("東海 4 県の合計面積:", round(tokai.area / 1e6, 1), "km²")
print("図形の種類:", tokai.geom_type)

In [ ]:
# 30 km 圏に入る都市
in_circle = pts_m[pts_m.within(circle)]
print(in_circle["city"].tolist())

### 練習問題 7

1. 浜松市から半径 50 km の圏内に入る都市を求めてください。
2. 名古屋市の 30 km 圏と浜松市の 50 km 圏が重なる面積（km²）を計算してください（`intersection` と `area`）。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
hamamatsu = pts_m.loc[pts_m["city"] == "浜松市", "geometry"].iloc[0]
circle_h = hamamatsu.buffer(50_000)
print(pts_m[pts_m.within(circle_h)]["city"].tolist())

# 2
print(round(circle.intersection(circle_h).area / 1e6, 1), "km²")
```

</details>

---
## 8. matplotlib による地図（コロプレス図）

GeoDataFrame の `plot()` は matplotlib で地図を描きます。`column=` に数値列を指定すると、
値に応じて色分けした **コロプレス図（塗り分け地図）** になります。

In [ ]:
ax = gdf.plot(figsize=(7, 6), color="lightblue", edgecolor="black")
ax.set_title("東海 4 県の県境")
ax.set_xlabel("経度")
ax.set_ylabel("緯度")
plt.show()

In [ ]:
# 人口密度のコロプレス図 + 県名ラベル
ax = gdf.plot(column="density", cmap="YlOrRd", legend=True, edgecolor="black", figsize=(8, 6),
              legend_kwds={"label": "人口密度（人/km²）"})
for name, p in zip(gdf["nam_ja"], gdf.representative_point()):
    ax.annotate(name, (p.x, p.y), ha="center", fontsize=11)
ax.set_title("東海 4 県の人口密度")
plt.show()

In [ ]:
# 県境の上に都市を重ねる（点の大きさは都市人口）
ax = gdf.plot(color="whitesmoke", edgecolor="gray", figsize=(8, 6))
pts.plot(ax=ax, color="red", markersize=pts["city_pop"] / 10_000, alpha=0.6)
for _, row in pts.iterrows():
    ax.annotate(row["city"], (row["lon"], row["lat"]), fontsize=9, xytext=(3, 3), textcoords="offset points")
ax.set_title("主要都市（円の大きさ = 人口）")
plt.show()

### 練習問題 8

1. `gdp_per_capita`（1 人あたり県内総生産）で色分けしたコロプレス図を描いてください（カラーマップは `"Blues"`）。
2. 上の地図に、名古屋市の 30 km 圏（`circle` を経度・緯度に戻したもの）を重ねてください
   （ヒント: `gpd.GeoSeries([circle], crs="EPSG:6674").to_crs(epsg=4326).plot(ax=ax, ...)`）。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```python
# 1
ax = gdf.plot(column="gdp_per_capita", cmap="Blues", legend=True, edgecolor="black", figsize=(8, 6),
              legend_kwds={"label": "1 人あたり県内総生産（万円）"})
ax.set_title("1 人あたり県内総生産")
plt.show()

# 2
ax = gdf.plot(color="whitesmoke", edgecolor="gray", figsize=(8, 6))
gpd.GeoSeries([circle], crs="EPSG:6674").to_crs(epsg=4326).plot(ax=ax, color="orange", alpha=0.4)
pts.plot(ax=ax, color="red")
ax.set_title("名古屋市の 30 km 圏")
plt.show()
```

</details>

---
## 9. folium による対話的な地図

folium は、Web 地図ライブラリ Leaflet を Python から使うためのライブラリです。
ズーム・ドラッグができ、マーカーや塗り分けを重ねられます。地図オブジェクトをセルの最後に置くか
`display(m)` とすると表示されます。

In [ ]:
# 基本の地図：中心（緯度, 経度）とズームレベルを指定
m = folium.Map(location=[35.1, 137.2], zoom_start=8)
m

In [ ]:
# マーカーとポップアップ・ツールチップ
m = folium.Map(location=[35.1, 137.2], zoom_start=8)
for _, row in pts.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f'{row["city"]}: 人口 {row["city_pop"]:,} 人',
        tooltip=row["city"],
    ).add_to(m)
m

In [ ]:
# 円マーカー：半径を人口に比例させる
m = folium.Map(location=[35.1, 137.2], zoom_start=8, tiles="CartoDB positron")
for _, row in pts.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=row["city_pop"] ** 0.5 / 60,
        color="crimson", fill=True, fill_opacity=0.5,
        tooltip=f'{row["city"]} {row["city_pop"]:,} 人',
    ).add_to(m)
m

In [ ]:
# GeoJson レイヤー：県境を描き、ツールチップで属性を表示
m = folium.Map(location=[35.1, 137.2], zoom_start=8)
folium.GeoJson(
    gdf[["nam_ja", "population", "geometry"]],
    style_function=lambda feature: {"fillColor": "#3388ff", "color": "black", "weight": 1, "fillOpacity": 0.2},
    tooltip=folium.GeoJsonTooltip(fields=["nam_ja", "population"], aliases=["県", "人口"]),
).add_to(m)
m

### コロプレス図（folium.Choropleth）

`folium.Choropleth` には、**図形（geo_data）** と **値の表（data）** を別々に渡します。
`data` には geometry を含まない普通の DataFrame を渡し、`columns=[キー列, 値列]` と
`key_on="feature.properties.キー"` で図形と値を対応付けます。

In [ ]:
m = folium.Map(location=[35.1, 137.2], zoom_start=8)
folium.Choropleth(
    geo_data=json.loads(gdf[["nam_ja", "geometry"]].to_json()),   # 図形（GeoJSON の辞書）
    data=gdf[["nam_ja", "density"]],                                # 値の表（geometry なし）
    columns=["nam_ja", "density"],
    key_on="feature.properties.nam_ja",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.5,
    legend_name="人口密度（人/km²）",
).add_to(m)
folium.LayerControl().add_to(m)
m

### 練習問題 9

1. `gdp_per_capita` を使ったコロプレス図を folium で描いてください（`fill_color="Blues"`）。
2. その地図に、各県の重心（`centroids`）へ県名をツールチップにしたマーカーを追加してください。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```python
m = folium.Map(location=[35.1, 137.2], zoom_start=8)
folium.Choropleth(
    geo_data=json.loads(gdf[["nam_ja", "geometry"]].to_json()),
    data=gdf[["nam_ja", "gdp_per_capita"]],
    columns=["nam_ja", "gdp_per_capita"],
    key_on="feature.properties.nam_ja",
    fill_color="Blues", fill_opacity=0.7, line_opacity=0.5,
    legend_name="1 人あたり県内総生産（万円）",
).add_to(m)
for name, c in zip(gdf["nam_ja"], centroids):
    folium.Marker([c.y, c.x], tooltip=name).add_to(m)
m
```

</details>

---
## 10. GeoJSON の書き出しと読み込み

分析結果（属性を追加した GeoDataFrame）は `to_json()` で GeoJSON 文字列にでき、ファイルに保存できます。
保存したファイルは左のファイルブラウザに現れ、`read_file()` で読み直せます。

In [ ]:
out = gdf[["nam_ja", "population", "area_km2", "density", "geometry"]]
with open("tokai4_stats.geojson", "w", encoding="utf-8") as f:
    f.write(out.to_json())
print("tokai4_stats.geojson を書き出しました")

In [ ]:
back = gpd.read_file("tokai4_stats.geojson")
print(back.drop(columns="geometry").round(1))
print(back.crs)

---
## まとめ

| トピック | 主な関数・操作 |
|---|---|
| 読み込み | `gpd.read_file()`, `.crs`, `.geometry`, `.geom_type`, `.bounds` |
| 属性の結合 | `GeoDataFrame.merge(df, on=...)` |
| 座標参照系 | `.to_crs(epsg=6674)`（メートル）, `.area`, `.distance()` |
| 重心・代表点 | `.centroid`（投影後）, `.representative_point()` |
| 点データ | `gpd.points_from_xy()`, `gpd.GeoDataFrame(df, geometry=..., crs=...)` |
| 空間結合 | `gpd.sjoin(left, right, predicate="within")` |
| 空間演算 | `.buffer()`, `.intersects()`, `.intersection()`, `.within()`, `.union_all()` |
| 静的な地図 | `gdf.plot(column=..., cmap=..., legend=True)`, `ax.annotate()` |
| 対話的な地図 | `folium.Map`, `Marker`, `CircleMarker`, `GeoJson`, `Choropleth`, `LayerControl` |
| 書き出し | `.to_json()` → ファイル保存, `read_file()` で再読込 |

## 次のステップ
- `python/folium/folium_beginner_tutorial.ipynb`, `folium_tokai_geojson_tutorial.ipynb` — folium の詳しい使い方
- `python/pandas/pandas_intermediate_tutorial.ipynb` — groupby・結合の復習
- `exercises/geo_visualization_exercises_10.ipynb` — 地理データ可視化の練習問題

---
## 総合演習：東海 4 県の地域経済マップ

次の課題に取り組んでください。

1. `gdf` に「1 人あたり県内総生産（`gdp_per_capita`）」と「人口密度（`density`）」の **順位** 列を追加し、県名・両指標・順位を表示する。
2. 都市データ `pts` を県に空間結合し、県ごとの都市人口合計が県人口に占める割合（%）を求める。
3. 名古屋市から各都市までの距離（km）を求め、近い順に表示する（投影座標系で計算）。
4. folium で、人口密度のコロプレス図の上に各都市の円マーカー（半径は人口に比例、ツールチップは都市名と距離）を重ねた地図を作る。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

In [ ]:
# 1. 順位列
rank = gdf[["nam_ja", "gdp_per_capita", "density"]].copy()
rank["gdp_rank"] = rank["gdp_per_capita"].rank(ascending=False).astype(int)
rank["density_rank"] = rank["density"].rank(ascending=False).astype(int)
print(rank.round(0))

# 2. 都市人口が県人口に占める割合
joined = gpd.sjoin(pts, gdf[["nam_ja", "population", "geometry"]], how="left", predicate="within")
ratio = joined.groupby("nam_ja").agg(city_pop=("city_pop", "sum"), population=("population", "first"))
ratio["ratio_pct"] = (ratio["city_pop"] / ratio["population"] * 100).round(1)
print(ratio)

# 3. 名古屋からの距離
pts_m = pts.to_crs(epsg=6674)
nagoya = pts_m.loc[pts_m["city"] == "名古屋市", "geometry"].iloc[0]
pts["dist_km"] = (pts_m.distance(nagoya) / 1000).round(1)
print(pts[["city", "dist_km"]].sort_values("dist_km"))

# 4. folium の地図
m = folium.Map(location=[35.1, 137.2], zoom_start=8, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=json.loads(gdf[["nam_ja", "geometry"]].to_json()),
    data=gdf[["nam_ja", "density"]], columns=["nam_ja", "density"],
    key_on="feature.properties.nam_ja", fill_color="YlOrRd", fill_opacity=0.6, line_opacity=0.5,
    legend_name="人口密度（人/km²）",
).add_to(m)
for _, row in pts.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]], radius=row["city_pop"] ** 0.5 / 60, color="navy", fill=True, fill_opacity=0.5,
        tooltip=f'{row["city"]}（名古屋から {row["dist_km"]} km）',
    ).add_to(m)
m

お疲れさまでした！ GeoPandas で「表として扱う」「投影して測る」「位置で結合する」、folium で「対話的に見せる」
という地理空間データ分析の基本の流れを身につけました。自分の関心のある地域データ（市区町村の GeoJSON など）を
アップロードして、同じ手順を試してみてください。